## Building LLM Call

In [161]:
# Start with import - ask the Cursor agent to explain any package you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [162]:
# Always do this first!!!
load_dotenv(override = True)

True

In [163]:
# Print the key prefixes to help with any debugging.

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GEMINI_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")

if deepseek_api_key:
    print(f"Deepseek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("Deepseek API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else: 
    print("Grok API Key not set")

if openrouter_api_key:
    print(f"Openrouter API Key exists and begins {openrouter_api_key [:6]}")
else:
    print("Openrouter API Key not set")




OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ
Deepseek API Key exists and begins sk-
Groq API Key exists and begins gsk_
Grok API Key exists and begins xai-
Openrouter API Key exists and begins sk-or-


In [164]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can adk a number of LLM's to evaluate their intelligence. 
Not a mathematical puzzle, but more of a thought-provoking question that 
requires intelligent insight. Include in your question that
the answer must be short
"""

request += "Answer only with the question. no explanation."
messages = [{"role": "user", "content": request}]

In [165]:
messages

[{'role': 'user',
  'content': "\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can adk a number of LLM's to evaluate their intelligence. \nNot a mathematical puzzle, but more of a thought-provoking question that \nrequires intelligent insight. Include in your question that\nthe answer must be short\nAnswer only with the question. no explanation."}]

### Step 1 logic

`request` is a prompt asking an LLM to invent a hard, subjective question we can pose to *other* LLMs as an intelligence test. We wrap it in the standard OpenAI chat format: a list of `messages`, each a dict with `role` ("user"/"assistant"/"system") and `content` (the text). This `messages` list is what actually gets sent to `chat.completions.create` below.

In [166]:
# Call gpt-5.4-mini to actually generate the question text.
# response.choices[0].message.content is where the model's reply text lives
# in the OpenAI chat completions response shape.

openai = OpenAI()
response = openai.chat.completions.create(model = "gpt-5.4-mini", messages = messages)
question = response.choices[0].message.content
display(Markdown(question))

You are advising a city that wants to reduce traffic congestion, pollution, and commute time without building new roads, expanding public transit, or using congestion pricing. What single policy would you recommend, and why? Keep your answer to one or two sentences.

## Calling LLMs From Multiple Providers

In [167]:
# OpenAI Compatible URLs
# Many providers expose an OpenAI-compatible /chat/completions endpoint.
# That means we can keep using the same `OpenAI` client class for all of them —
# we just point `base_url` at the provider and pass that provider's API key.
# This is why the rest of the notebook can treat every model the same way.

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROK_BASE_URL = "https://api.x.ai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [168]:
# OpenAI client libraries with the right base_url and key.
# If this surprises you, please see Guide 9 in the Guides folder!

anthropic = OpenAI(api_key = anthropic_api_key, base_url = ANTHROPIC_BASE_URL)
deepseek = OpenAI(api_key = deepseek_api_key, base_url = DEEPSEEK_BASE_URL)
gemini = OpenAI(api_key = google_api_key, base_url = GEMINI_BASE_URL)
groq = OpenAI(api_key = groq_api_key, base_url = GROQ_BASE_URL)
grok = OpenAI(api_key = grok_api_key, base_url = GROK_BASE_URL)
openrouter = OpenAI(api_key = openrouter_api_key, base_url = OPENROUTER_BASE_URL)
ollama = OpenAI(base_url = OLLAMA_BASE_URL, api_key = "ollama")

In [169]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]


### Step 2 logic

Now we ask every provider the *same* `question` and collect the results in two parallel lists: `competitors` (model names) and `answers` (their responses), via the `record()` helper below. `messages` here is reused across every model call — same question, different model each time.

In [170]:
# Small helper so each model-call cell below doesn't repeat this bookkeeping:
# track which model answered (competitors), store its answer (answers),
# and immediately render it so we can eyeball it as we go.

def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [171]:
# The API we know well.
# Reasoning effect can be none, low, medium, high, or xhigh.

model_name = "gpt-5.4-nano"

response = openai.chat.completions.create(model = model_name,
                                          messages = messages,
                                          reasoning_effort = "none")
answer = response.choices[0].message.content

record(model_name, answer)



Adopt citywide employer “work-from-home/remote-work” mandates or strong incentives (e.g., requiring firms to allow at least 2–3 days/week remote work for eligible roles). This directly reduces peak-hour vehicle trips, cutting congestion and emissions while shortening commute demand—without needing new roads, transit expansion, or congestion-pricing.

In [172]:
model_name = "claude-sonnet-4-6"

response = anthropic.chat.completions.create(model = model_name,
                                             messages = messages)
answer = response.choices[0].message.content
record(model_name, answer)

I would recommend **mandatory staggered work hours** across major employers, shifting start times so not everyone commutes simultaneously—this directly reduces peak-hour congestion, cutting pollution and commute times without new infrastructure or fees, while requiring only regulatory coordination rather than capital investment.

In [173]:
model_name = "gemini-3.1-flash-lite"
response = gemini.chat.completions.create(model = model_name,
                                          messages = messages)
answer = response.choices[0].message.content
record(model_name, answer)


I recommend implementing citywide **flexible or staggered work hours**, which flattens the peak-hour demand curve and spreads existing traffic volume more evenly across the day. This reduces gridlock and idle-engine emissions without requiring infrastructure investment or direct financial penalties.

In [174]:
model_name = "deepseek-v4-flash"
response = deepseek.chat.completions.create(model = model_name,
                                            messages = messages)
answer = response.choices[0].message.content
record(model_name, answer)

Mandate that all employers allow remote work for at least three days per week for roles that can be performed remotely, which directly reduces vehicle trips, cuts emissions, and shortens commute times for those who still drive during off-peak hours.

In [175]:
model_name = "openai/gpt-oss-120b"
response = groq.chat.completions.create(model = model_name,
                                        messages = messages)
answer = response.choices[0].message.content
record(model_name, answer)

Implement a citywide mandate for flexible work hours and remote‑work options, shifting a large share of commuting off peak periods and reducing overall vehicle trips. This cuts congestion, emissions, and average travel times without adding infrastructure or pricing mechanisms.

In [176]:
model_name = "moonshotai/kimi-k2.6"
response = openrouter.chat.completions.create(model = model_name,
                                              messages = messages)
answer = response.choices[0].message.content
record(model_name, answer)

 Incentivize—or, where legally possible, mandate—remote or hybrid work for all compatible jobs by offering employers tax credits or priority permitting; this immediately takes cars off the road, which slashes tailpipe emissions, shortens commutes for remaining drivers, and erases travel time entirely for teleworkers without new infrastructure, expanded transit, or road pricing.

## For the next call we'll use Ollama

In [177]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [178]:
import requests
requests.get("http://localhost:11434").content


b'Ollama is running'

In [179]:

models = requests.get("http://localhost:11434/v1/models").json()
for model in models.get("data"):
    print(model.get("id"))

llama3.2:latest
qwen3:4b
qwen3.6:latest


In [180]:
model_name = "llama3.2:latest"
response = ollama.chat.completions.create(model = model_name,
                                         messages = messages)
answer = response.choices[0].message.content
record(model_name, answer)

I would recommend implementing flexible work-from-home (WFH) policies with paid leave for daily commutes, enabling employees to reduce the number of daily drivers and associated emissions, while also promoting a healthier work-life balance. This policy would require government support for employers that want to adopt WFH arrangements, as well as investments in digital infrastructure for remote workers.

In [181]:
model_name = "qwen3:4b"
response = ollama.chat.completions.create(model = model_name,
                                          messages = messages)
answer = response.choices[0].message.content
record(model_name, answer)

Incentivizing remote work and flexible schedules to reduce single-vehicle commutes directly cuts pollution, congestion, and commute time without new infrastructure or pricing mechanisms, as it decreases the number of cars traveling during peak hours through behavior change.

In [182]:
# So where are we
print(len(competitors))
print(competitors)
print(answers)

8
['gpt-5.4-nano', 'claude-sonnet-4-6', 'gemini-3.1-flash-lite', 'deepseek-v4-flash', 'openai/gpt-oss-120b', 'moonshotai/kimi-k2.6', 'llama3.2:latest', 'qwen3:4b']
['Adopt citywide employer “work-from-home/remote-work” mandates or strong incentives (e.g., requiring firms to allow at least 2–3 days/week remote work for eligible roles). This directly reduces peak-hour vehicle trips, cutting congestion and emissions while shortening commute demand—without needing new roads, transit expansion, or congestion-pricing.', 'I would recommend **mandatory staggered work hours** across major employers, shifting start times so not everyone commutes simultaneously—this directly reduces peak-hour congestion, cutting pollution and commute times without new infrastructure or fees, while requiring only regulatory coordination rather than capital investment.', 'I recommend implementing citywide **flexible or staggered work hours**, which flattens the peak-hour demand curve and spreads existing traffic vo

In [183]:
# it's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n Answer: {answer}\n\n")

Competitor: gpt-5.4-nano

 Answer: Adopt citywide employer “work-from-home/remote-work” mandates or strong incentives (e.g., requiring firms to allow at least 2–3 days/week remote work for eligible roles). This directly reduces peak-hour vehicle trips, cutting congestion and emissions while shortening commute demand—without needing new roads, transit expansion, or congestion-pricing.


Competitor: claude-sonnet-4-6

 Answer: I would recommend **mandatory staggered work hours** across major employers, shifting start times so not everyone commutes simultaneously—this directly reduces peak-hour congestion, cutting pollution and commute times without new infrastructure or fees, while requiring only regulatory coordination rather than capital investment.


Competitor: gemini-3.1-flash-lite

 Answer: I recommend implementing citywide **flexible or staggered work hours**, which flattens the peak-hour demand curve and spreads existing traffic volume more evenly across the day. This reduces gri

In [184]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index + 1}\n\n"
    together += answer +"\n\n"


### Step 3 logic

`together` concatenates every competitor's answer into one big labeled block ("Response from competitor N"), because the judge model in Step 4 needs to see all the answers in a single prompt to compare and rank them.

In [185]:
print(together)

# Response from competitor 1

Adopt citywide employer “work-from-home/remote-work” mandates or strong incentives (e.g., requiring firms to allow at least 2–3 days/week remote work for eligible roles). This directly reduces peak-hour vehicle trips, cutting congestion and emissions while shortening commute demand—without needing new roads, transit expansion, or congestion-pricing.

# Response from competitor 2

I would recommend **mandatory staggered work hours** across major employers, shifting start times so not everyone commutes simultaneously—this directly reduces peak-hour congestion, cutting pollution and commute times without new infrastructure or fees, while requiring only regulatory coordination rather than capital investment.

# Response from competitor 3

I recommend implementing citywide **flexible or staggered work hours**, which flattens the peak-hour demand curve and spreads existing traffic volume more evenly across the day. This reduces gridlock and idle-engine emissions

In [186]:
judge = f""" 
You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["Best competitor number", 
             "Second best competitor number", 
             "Third best competitor number", ...]}}
Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the 
competitors, nothing else. Do not include markdown
formatting or code blocks.
"""

### Step 4 logic

`judge` is the prompt that turns an LLM into a grader: it includes the original `question`, all the collected answers (`together`), and strict instructions to reply with *only* JSON in the shape `{"results": [...]}` — a ranked list of competitor numbers, best first. The strict "JSON only, no markdown" instruction matters because we `json.loads()` the raw response text in Step 5 — any stray prose or code-fence would break the parse (this is exactly what caused the earlier `JSONDecodeError`: a typo in this template produced invalid JSON).

In [187]:
print(judge)

 
You are judging a competition between 8 competitors.
Each model has been given this question:

You are advising a city that wants to reduce traffic congestion, pollution, and commute time without building new roads, expanding public transit, or using congestion pricing. What single policy would you recommend, and why? Keep your answer to one or two sentences.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["Best competitor number", 
             "Second best competitor number", 
             "Third best competitor number", ...]}
Here are the responses from each competitor:

# Response from competitor 1

Adopt citywide employer “work-from-home/remote-work” mandates or strong incentives (e.g., requiring firms to allow at least 2–3 days/week remote work for eligible roles). This directly reduces peak-hour vehicle trips, cutting congestion and em

In [188]:
judge_messages = [{"role":"user", "content":judge}]

## And now for Grok


In [189]:
# Judgement time!
# Grok is "The most truth-seeking large language model
# in the world"

model_name = "grok-4.3"
response = grok.chat.completions.create(model = model_name,
                                        messages = judge_messages)
results = response.choices[0].message.content
print(results)

{"results": ["6", "1", "4", "5", "3", "2", "8", "7"]}


In [190]:
# Ok lets turn this into results!

results_dict = json.loads(results)
rank = results_dict["results"]
for index, result in enumerate(rank):
    competitor = competitors[int(result) -1]
    print(f"Rank {index +1}: {competitor}")

Rank 1: moonshotai/kimi-k2.6
Rank 2: gpt-5.4-nano
Rank 3: deepseek-v4-flash
Rank 4: openai/gpt-oss-120b
Rank 5: gemini-3.1-flash-lite
Rank 6: claude-sonnet-4-6
Rank 7: qwen3:4b
Rank 8: llama3.2:latest


### Step 5 logic

`json.loads(results)` parses the judge's JSON string into a Python dict; `results_dict["results"]` pulls out the ranked list of competitor numbers (as strings, 1-indexed — hence `int(result) - 1` to index into the 0-indexed `competitors` list). The loop just prints each competitor next to its rank.